<a href="https://colab.research.google.com/github/facundoVega/ProcesamientoDelHabla/blob/main/TP3-Chatbot/Horacio_Facundo_Vega_TP3_chatbot_retrieval_template_parte_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP3 Chatbots basados en recuperación de la información

En inglés information retrieval chatbots

# Motor de búsqueda

* Búsqueda por palabras clave: Extrae palabras clave de la pregunta del usuario y busca coincidencias en las preguntas almacenadas.

* Similitud del coseno: Si has representado las preguntas como vectores (por ejemplo, usando TF-IDF o word embeddings), puedes usar la similitud del coseno para medir la distancia entre las preguntas.

* Embeddings: Utiliza modelos de word embeddings como por ejemplo Word2Vec para obtener representaciones semánticas de las preguntas y las consultas del usuario.

## Librerías

Debes trabajar en Python. Puedes usar las librerías sklearn, pandas, spacy o nltk o gensim para el punto de usar o buscar embeddings.

In [ ]:
!pip install spacy --quiet
!python -m spacy download es_core_news_sm --quiet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Actividades



### 1) Elaborar un dataset de preguntas y respuestas para crear un Chatbot para un aplicación particular. ( 3 puntos )

1.1 Debe definir la aplicación (atención al cliente bancario, atención a estudiantes universitarios, etc).

1.2 El listado de preguntas y respuestas debe tener como mínimo 20 elementos pregunta - respuesta.

###  2) Crear el chatbot utilizando TFIDF y similitud del coseno. (1 punto)

### 3) Crear otro chatbot utilizando embeddings. Indique cuál embedding (1 punto) pre-entrenado de las librerías Spacy o nltk o gensim eligió. Justifique.

### 4) Muestra ambos chatbots funcionando (1 punto)

Adjuntar la lista de preguntas y respuestas utilizadas para probar el funcionamiento.

Releve o indique cuáles respondió correctamente y cuáles no.

### 5) Añade tus conclusiones de todo lo realizado (2 punto)

* Resalte e indique en cuáles respuestas falla o tiene problemas.
* Cuáles preguntas confunde.
* Compare los resultados de los chatbots.



### No olvides:

* Explicar tus decisiones y configuraciones. Añadir tus conclusiones.
* Anunciar en el foro cuál será tu aplicación y postear tu entrega y tus avances.
* Debes subir tu notebook a un repo GitHub público de tu propiedad compartido + enlace colab.
* Documentar todo el proceso.
* Citar tus fuentes





# Resolución del TP

---



### 1) Elaborar un dataset de preguntas y respuestas para crear un Chatbot para un aplicación particular.

La aplicación sera un asistente virtual para  una tienda de cómics y novelas gráficas:

**Objetivos:**
*   Recomendar cómics.
*   Ayudar a principiantes.
*   Resolver dudas frecuentes de clientes.
*   Informar sobre stocks, pedidos y envíos.






**Listado de preguntas y respuestas**

In [ ]:
import pandas as pd

In [ ]:
#Cargo las preguntas.
import requests

url_preguntas = "https://raw.githubusercontent.com/facundoVega/ProcesamientoDelHabla/refs/heads/main/TP3-Chatbot/chatbot-data/preguntas.txt"

preguntas = requests.get(url_preguntas).text.splitlines()

print(f"Lista de {len(preguntas)} preguntas:")
print("*********************")
for pregunta in preguntas:
  print(pregunta)


In [ ]:
#Cargo las respuestas.
import requests

url_respuestas = "https://raw.githubusercontent.com/facundoVega/ProcesamientoDelHabla/refs/heads/main/TP3-Chatbot/chatbot-data/respuestas.txt"

respuestas = requests.get(url_respuestas).text.splitlines()

print(f"Lista de {len(respuestas)} respuestas:")
print("*********************")
for respuesta in respuestas:
  print(respuesta)

###  2) Crear el chatbot utilizando TFIDF y similitud del coseno

Antes de crear el chatbot eliminare las stop words y los signos de puntuación de la base de preguntas.

**Aclaración Importante:**


*Se procede a realizar este preprocesamiento de las preguntas por que la idea es comparar el funcionamiento de ambos chabots: el basado en TF-IDF y el basado en Embeddings. Como en el chatbot basado en Embeddings se eliminaran stop words y signos de puntuación antes de calcular el vector promedio de cada pregunta, se hace este mismo preprocesamiento en el chatbot basado en TF-IDF, de esta manera tener una comparación mas homogenea entre estos chatbots.*

In [ ]:
!python -m spacy download es_core_news_md

In [ ]:
import spacy

nlp = spacy.load("es_core_news_md")

In [ ]:
#Funcion que recibe una pregunta y elimina stop words y signos de puntuación
def limpiar_pregunta(pregunta):
    doc = nlp(pregunta)

    return " ".join(
        token.text
        for token in doc
        if not token.is_stop and not token.is_punct
    )

In [ ]:
#pruebo la función
pregunta = "¿Quedan comics de Batman disponibles?"
print(limpiar_pregunta(pregunta))

Limpio todas las preguntas utilizadas para el chatbot

In [ ]:
preguntas_limpias = []

for pregunta in preguntas:
  preguntas_limpias.append(limpiar_pregunta(pregunta))


Procedo a vectorizar mis preguntas con el modelo TfidVectorizer, el cual nos devolvera un vector por cada una de nuestras preguntas.

In [ ]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(preguntas_limpias)

tfidf_matrix

**Importante:**


*Para facilitar la comparación de ambos chatbots, requerida en las consignas posteriores del TP, a parte de la respuesta como salida de este método se incluira:

* *La similitud coseno de la pregunta ganadora.*
* *La pregunta misma de la base del chatbot que fue considerada como la más parecida a la realizada por el usuario.

*En un chatbot real estas salidas no seráan necesarias, sin embargo, para el análisis posterior nos sera sumamente útiles.*

Creo una función para encontrar la pregunta cargada en el chatbot mas parecida a la pregunta realizada por el usuario, utilizando la similitud coseno.

In [ ]:
def responder_pregunta_tf_idf(consulta_usuario):
    consulta_vec = vectorizer.transform([limpiar_pregunta(consulta_usuario)])

    similitudes = cosine_similarity(consulta_vec, tfidf_matrix).flatten()

    indice_mejor_coincidencia = similitudes.argmax()

    return (
        preguntas[indice_mejor_coincidencia],
        respuestas[indice_mejor_coincidencia],
        similitudes[indice_mejor_coincidencia]
    )


In [ ]:
#Pruebla simple de la función

pregunta_usuario = "¿Qué cómic me recomiendan para empezar a leer?"

pregunta_encontrada, respuesta, similitud = (
    responder_pregunta_tf_idf(pregunta_usuario)
)

print("Pregunta usuario: ", pregunta_usuario)
print("Pregunta encontrada: ", pregunta_encontrada)
print("RESPUESTA: ", respuesta )
print("SIMILITUD COSENO: ", similitud)

### 3) Crear otro chatbot utilizando embeddings. Indique cuál embedding pre-entrenado de las librerías Spacy o nltk o gensim eligió. Justifique.

El modelo que voy a utilizar para el chatbot de embedding es: "**es_core_news_md**".


---



* Utilizaré este modelo por que esta específicamente entrenado para el idioma español y ya incluye embeddings preentrenados de palabras que permiten representar el significado de los textos.
* También por que es relativamente ligero para ejecutarlo en el entorno de colab, y su integración con Spacy facilita el preprocesamiento y el cálculo de la similitud semántica.

Se descartaron otros modelos como Word2Vec o Glove de Gensim debido a que requieren mucho mas pasos para obtener vectores de frases completas, la cual es una de las tareas claves en el desarrollo de mi chatbot.

**Aclaración importante**

*La difencia principal a la hora de desarrollar el chatbot con Embeddings es que cada palabra de la pregunta va a tener un vector. En cambio con TF-IDF cada pregunta genera un vector donde cada posición del vector es una palabra de la pregunta.  Por eso para obtener el vector de cada pregunta utilizando el enfoque de embeddings calcularemos el promedio de vectores que conforma cada pregunta.*

Desarrollo una función que recibe una pregunta o texto y devuelve un vector que representa el promedio de todos los vectores que conforman el texto recibido por parámetro.

In [ ]:
import numpy as np

def texto_a_vector(texto):

  doc = nlp(texto)

  vectores = [
      token.vector
      for token in doc
      if not token.is_stop
      and not token.is_punct
      and token.has_vector
  ]

  return np.mean(vectores, axis=0)


In [ ]:
#pruebo la funcion anterior
v = texto_a_vector("¿Que cómics son los mas populares")
print(v.shape)

Como vemos se genero un vector de 300 dimensiones a partir del texto que enviamos a la función, por lo que podemos asumir que nuestra función nos esta dando el vector promedio a partir de un texto.

**Genero embeddings para cada una de las preguntas del chatbot.**

In [ ]:
preguntas_vectores = np.array([
    texto_a_vector(pregunta)
    for pregunta in preguntas
])

print(preguntas_vectores.shape)

Ahora desarrollo la función para responder preguntas con el enfoque embeddings:

In [ ]:
def responder_pregunta_embeddings(consulta_usuario):
    consulta_vec = texto_a_vector(consulta_usuario)

    similitudes = cosine_similarity(
        [consulta_vec],
        preguntas_vectores
        ).flatten()

    indice_mejor_coincidencia = similitudes.argmax()

    return (
        preguntas[indice_mejor_coincidencia],
        respuestas[indice_mejor_coincidencia],
        similitudes[indice_mejor_coincidencia]
    )


In [ ]:
#Pruebo la funcion
pregunta_prueba = "¿Como se manejan con el stock?"

pregunta_encontrada, respuesta, similitud = (
    responder_pregunta_embeddings(pregunta_prueba)
)

print("Pregunta usuario: ", pregunta_prueba)
print("Pregunta encontrada: ", pregunta_encontrada)
print("RESPUESTA: ", respuesta )
print("SIMILITUD COSENO: ", similitud)


### 4) Muestra ambos chatbots funcionando





In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

#caja de texto
pregunta_text = widgets.Text(
    placeholder = 'Escribe tu pregunta...',
    description = 'Pregunta:',
    layout= widgets.Layout(width='80%')
)

#botón
btn = widgets.Button(
    description = 'Preguntar',
    button_style = 'primary'
)

#output
output = widgets.Output()

def consultar_chatbots(b):
  with output:
      output.clear_output()

      pregunta = pregunta_text.value.strip()

      if not pregunta:
          print("Ingrese una pregunta.")
          return

      pregunta_tfidf, resp_tfidf, similitud_tfidf  =  (responder_pregunta_tf_idf(pregunta))
      preg_emb, resp_emb, similitud_emb = (responder_pregunta_embeddings(pregunta))


      display(HTML(f"""
      <h3>Pregunta</h3>
      <p>{pregunta}</p>

      <table border="1" style="width:100%; border-collapse:collapse;">
          <tr stlye="background-color:#f2f2f2">
            <th>TF-IDF</th>
            <th>Embeddings</th>
          </tr>
          <tr>
            <td style="vertical-align:top; padding:10px;">
                {resp_tfidf}
            </td>
            <td style="vertical-align:top; padding:10px;">
                { resp_emb}
            </td>
          </tr>
      </table>
      """))

btn.on_click(consultar_chatbots)

display(pregunta_text)
display(btn)
display(output)





**Adjuntar la lista de preguntas y respuestas utilizadas para probar el funcionamiento.**

---



Para probar el funcionamiento de los chatbots, voy a utilizar 25 preguntas de prueba con la siguiente estructura:

* 10 Preguntas idénticas a la base de preguntas de los chatbots.
* 10 preguntas reformuladas
* 5 Preguntas especiales no contempladas por el chatbot pero que esconden la intención de preguntas ya cargadas en la base de los chatbots.


**Las preguntas de prueba se encuentran en el siguiente link de github:**


[Preguntas-de-prueba](github.com/facundoVega/ProcesamientoDelHabla/blob/main/TP3-Chatbot/chatbot-data/preguntas-prueba.txt)

Aqui encontraran las 25 preguntas de prueba con la siguiente disposición:

* **Primeras 10 preguntas de la lista:** idénticas a la base de preguntas del chatbot para medir recuperación exacta
* **Segundas 10 preguntas de la lista:** Reformuladas, miden como se comporta el chatbot en cuanto a generalización semántica.
* **Ultimas 5 preguntas:** No contempladas, miden como se comporta el chatbot en estos escenarios.

**Las respuestas de los  chatbots se encuentran en el siguiente Link.**


[Respuestas-de-chatbots ](https://github.com/facundoVega/ProcesamientoDelHabla/blob/main/TP3-Chatbot/chatbot-data/respuestas.txt)

In [ ]:
#Importamos las preguntas para probar el chatbot


url_preguntas_prueba = "https://raw.githubusercontent.com/facundoVega/ProcesamientoDelHabla/main/TP3-Chatbot/chatbot-data/preguntas-prueba.txt"

preguntas_prueba = requests.get(url_preguntas_prueba).text.splitlines()

print(f"Lista de {len(preguntas_prueba)} preguntas de prueba:")
print("*********************")
for pregunta_prueba in preguntas_prueba:
  print(pregunta_prueba)

En mi repositorio de github tengo cargada las preguntas de prueba en una lista de objetos JSON que tienen las siguientes propiedades para cada pregunta de prueba:

* **pregunta**: la pregunta de prueba
* **tipo**: Base, Reformulada o Nueva,
* **esperada**: la pregunta de la base con la que se espera que la preguta nueva se asocie.

In [ ]:
#Importo las preguntas de prueba en format json
import requests

url = "https://raw.githubusercontent.com/facundoVega/ProcesamientoDelHabla/main/TP3-Chatbot/chatbot-data/preguntas-prueba.json"

casos_prueba = requests.get(url).json()


print(type(casos_prueba))
print(f"Casos cargados: {len(casos_prueba)}")
print(casos_prueba[0])




**Releve o indique de cada chatbot cuáles respondió correctamente y cuáles no.**

---



In [ ]:
import pandas as pd

resultados = []


for caso in casos_prueba:

  pregunta = caso["pregunta"]
  tipo = caso["tipo"]
  esperada = caso["esperada"]

  #TF-IDF
  pregunta_tfidf, respuesta_tfidf, similitud_tfidf = (
      responder_pregunta_tf_idf(pregunta)
  )

  #Embeddings
  pregunta_emb, respuesta_emb, similitud_emb = (
      responder_pregunta_embeddings(pregunta)
  )

  resultados.append({
      "Pregunta": pregunta,
      "Tipo": tipo,
      "Esperada": esperada,

      "TF-IDF Pregunta Recuperada":  pregunta_tfidf,
      "TF-IDF Coseno": similitud_tfidf,
      "TF-IDF Correcto": pregunta_tfidf.strip().lower() == esperada.strip().lower(),

      "Emeddings Pregunta Recuperada":  pregunta_emb,
      "Embeddings Coseno": similitud_emb,
      "Embeddings Correcto":  pregunta_emb.strip().lower() == esperada.strip().lower()
  })


df_resultados = pd.DataFrame(resultados)





In [ ]:
#Agrego iconos de tilde o de error en el dataframe para que la tabla quede mas facil de visualizar.
df_visual = df_resultados.copy()

df_visual["TF-IDF Correcto"] = (
    df_visual["TF-IDF Correcto"]
    .map({True: "✅", False: "❌"})
)

df_visual["Embeddings Correcto"] = (
    df_visual["Embeddings Correcto"]
    .map({True: "✅", False: "❌"})
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

print("Tabla completa")
print("***************")
display(df_visual)

### 5) Añade tus conclusiones de todo lo realizado



**Tabla Resumen**

In [ ]:
total = len(df_resultados)

resumen = pd.DataFrame({
    "Modelo": ["TF-IDF", "Embeddings"],

    "Aciertos": [
        df_resultados["TF-IDF Correcto"].sum(),
        df_resultados["Embeddings Correcto"].sum()
    ]
})

resumen["Errores"] = (
   total - resumen["Aciertos"]
)

resumen["Accuracy (%)"] = (
    resumen["Aciertos"] / total *100
).round(2)

print("Resumen General")
print("***************")
display(resumen)

* **Resalte e indique en cuáles respuestas falla o tiene problemas.**
* **Cuáles preguntas confunde.**

---



In [ ]:
resumen_tipo = []

for tipo in df_resultados["Tipo"].unique():

  subset = df_resultados[
      df_resultados["Tipo"] == tipo
  ]

  resumen_tipo.append({
      "Tipo": tipo,
      "TF-IDF Accuracy (%)": round(subset["TF-IDF Correcto"].mean()*100, 2),
      "Embeddings Accuracy (%)": round(subset["Embeddings Correcto"].mean()*100, 2)

  })

resumen_tipo = pd.DataFrame(resumen_tipo)
print("Resumen por tipo de pregunta de prueba")
print("**************************************")
display(resumen_tipo)


**Compare los resultados de los chatbots.**

---



**Comparación de resultados:**

* Ambos chatbots tuvieron una precisión del 100% en las preguntas que eran idénticas a las que estaban en su base de preguntas, lo cual era algo esperado pero confirma que ambos chatbots funcionan bien. De hecho la similitud coseno en todas estas 10 preguntas fueron de 1 lo cual la similitud es perfecta para ambos chatbots.


* Para el caso de las preguntas reformuladas ambas aplicaciones tuvieron un desempeño similar aunque el modelo de TF-IDF funcionó ligeramente mejor que el de embeddings. Sin embargo el porcentaje de precisión en esta categoría fue bajo en ambos casos.

* En cuanto a las preguntas nuevas, las cuales son preguntas reformuladas cuya intención subyacente es una pregunta de la base,ambos chatbots respondieron bien 3 de 5 preguntas.



**Conclusión:**


Como se destaca en la tabla de comparación los dos tuvieron una precisión similar tanto a nivel general como en cada categoría particular de las preguntas de prueba. Por ende, si tuviera que tomar una decisión de que chatbot usar solo utilizando este experimento, me quedaria con el de TF-IDF ya que a nivel general estos chatbots implican  menos costos computacional que los chatbots de Embeddings. Sin embargo cabe destacar que las preguntas que el chatbot de embedding recupero mal guardaban cierta coherencia y proximidad con la pregunta original. Lo cual da a entender que aunque la precision no fue del 100 % en todos los casos el chatbot de Embeddings logro capturar bastante bien la similitud semántica general.

# FUENTES:

---

Las fuentes utilizadas para el desarrollo de este TP fueron basicamente 2:

* En primer lugar se utilizo como base los colab compartidos por la docente en la explicación de chatbots de recuperación. Se uso ese código fuente como base y se añadio algunos detalles propios para poder hacer un análisis un poco mas minucióso del desempeño de cada modelo de chatbot.

* En segundo lugar, se trabajo con github copilot que por ser estar este en el contexto corporativo donde trabajo y debido a politicas de privacidad no puedo compartir los links de los chats. Tambien este trabajo se desarrollo en varios días, por lo que no habría un solo link de conversación. Por eso paso a detallar en donde pedi asistencia a copilot para que quede registrado:

  * Le pedí las preguntas y respuestas de la base del chatbot, y de la lista que me dio , refiné y me quede con las 20 mejores de las mismas. Lo mismo con las preguntas de prueba.

  * Le pedí ayuda con las tablas y con la interfaz para mostrar los chatbots funcionando.

*Todo el código presente en el chatbot fue transcrito por mi y guardo total responsabilidad por la funcionalidad o no de los mismos en la ejecución de este colab.*
